In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA A40


In [2]:
!pip install --upgrade pip
!pip install transformers accelerate bitsandbytes
!pip install hf_transfer
!pip install --upgrade transformers
!pip install tokenizers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 14.4 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.2
    Uninstalling pip-25.2:
      Successfully uninstalled pip-25.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 80.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 10.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 50.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 58.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 105.5 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.5/803.5 kB 16.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [accelerate]9 [accelerate]s]ub]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 53.8 MB/s  0:00:00


# Loading the model

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# -------------------------
# 1. SETUP: Correct Model Name and Loading
# -------------------------
# The correct, case-sensitive model ID from the Hugging Face Hub
model_name = "mistralai/Ministral-8B-Instruct-2410" 

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Note: Loading a 14B model requires significant VRAM (e.g., 24GB+)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    load_in_4bit=True  # Enables loading on smaller GPUs, if available
)

# -------------------------
# 2. PROMPT: Use the Correct Chat Format
# -------------------------
# Instruct models perform best when instructions are formatted into a "conversation"
# This ensures the model knows it should generate an assistant response.
user_prompt = "Hello, summarize the news in 2 sentences."

messages = [
    # Best practice to give the model context/persona
    {"role": "system", "content": "You are a concise AI assistant."}, 
    {"role": "user", "content": user_prompt}
]

# Apply the model's specific chat template (e.g., Mistral's format)
chat_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True # Tells the model it's the assistant's turn to respond
)

inputs = tokenizer(chat_prompt, return_tensors="pt").to(model.device)

# -------------------------
# 3. GENERATE: Output
# -------------------------
outputs = model.generate(**inputs, max_new_tokens=50)

# Decode only the generated response (excluding the input prompt)
prompt_len = inputs["input_ids"].shape[1]
generated_ids = outputs[0][prompt_len:]
output_text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

print(output_text)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.


config.json: 0.00B [00:00, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/1.07G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Sure, here's a concise summary of the latest news:

1. **SpaceX's Starship SN20 successfully completed its first orbital flight test, reaching an altitude of 10 km before returning to Earth.**
2. **The World Health


# Scraping

In [12]:
import re
import requests

# -------------------------
# 1. Input Car Data
# -------------------------
target_car_data = """
car type: Citroen Berlingo 1.5 Blue-HDi Club M 1000
Build year: 2019
First registration: 08/2019
Odometer reading: 115,077 km
"""

# -------------------------
# 2. Query Builder (Snippets Only)
# -------------------------
def build_search_query(target_car_data):
    text = target_car_data

    # Extract model keywords
    model_line = re.search(r"car type:\s*(.+)", text, re.IGNORECASE)
    model_keywords = model_line.group(1) if model_line else ""

    # Extract year
    year = re.search(r"Build year:\s*(\d{4})", text)
    year_val = year.group(1) if year else ""

    # Extract mileage
    km_match = re.search(r"Odometer reading:\s*([\d, ]+)\s*km", text)
    if km_match:
        exact_km = int(km_match.group(1).replace(",", "").replace(" ", ""))
        mileage_range = f"{exact_km - 2000}-{exact_km + 2000} km"
    else:
        mileage_range = ""

    # Build query for snippets only
    query = (
        f'"{model_keywords}" {year_val} "{mileage_range}" '
        f'price site:autoscout24.com OR site:mobile.de'
    )

    return query.strip()

# -------------------------
# 3. TAVILY Search (Snippets Only)
# -------------------------
TAVILY_KEY = "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"
TAVILY_URL = "https://api.tavily.com/search"

search_query = build_search_query(target_car_data)
print("🔍 Generated Query:")
print(search_query)

payload = {
    "api_key": TAVILY_KEY,
    "query": search_query,
    "search_depth": "basic",  # basic = snippet only
    "max_results": 20,
    "include_answer": False
}

response = requests.post(TAVILY_URL, json=payload)
results = response.json().get("results", [])

print(f"\nFound {len(results)} listings.\n")

# -------------------------
# 4. Extract price from snippet
# -------------------------
def extract_price(text):
    """
    Looks for € or EUR prices inside snippet.
    Works for formats like:
    - €12,900
    - 12.900 EUR
    - 12900 €
    """
    price_regex = r"(€\s?\d[\d.,]+|\d[\d.,]+\s?€|\d[\d.,]+\s?EUR)"
    match = re.search(price_regex, text)
    if not match:
        return None

    # normalize
    price = match.group(0)
    price_num = re.sub(r"[^\d.]", "", price)

    try:
        return float(price_num.replace(",", "").replace(".", ""))
    except:
        return price

# -------------------------
# 5. Display results with prices
# -------------------------
for i, r in enumerate(results):
    snippet = r.get("snippet", "")
    price = extract_price(snippet)

    print(f"--- LISTING {i+1} ---")
    print("Title:", r.get("title"))
    print("URL:", r.get("url"))
    print("Snippet:", snippet)
    print("Extracted Price:", price)
    print()


🔍 Generated Query:
"Citroen Berlingo 1.5 Blue-HDi Club M 1000" 2019 "113077-117077 km" price site:autoscout24.com OR site:mobile.de

Found 7 listings.

--- LISTING 1 ---
Title: Find Citroen Berlingo from 2019 for sale - AutoScout24
URL: https://www.autoscout24.com/lst/citroen/berlingo/re_2019
Snippet: 
Extracted Price: None

--- LISTING 2 ---
Title: Citroën Berlingo von 2019 gebraucht kaufen bei mobile.de Used Citroen Berlingo for sale - AutoScout24 Citroën Berlingo Diesel gebraucht kaufen bei mobile.de Citroën Berlingo Kombi Selection Diesel gebraucht kaufen bei ... Find used Citroen Berlingo in 06749-bitterfeld- (bitterfeld ... Citroën Berlingo Kastenwagen gebraucht kaufen bei mobile . de Citroën Berlingo Diesel gebraucht kaufen bei mobile . de Citroën Berlingo 2019 gebraucht kaufen bei mobile . de Citroën Berlingo Kastenwagen gebraucht kaufen bei mobile.de
URL: https://suchen.mobile.de/auto/citroen-berlingo-2019.html
Snippet: 
Extracted Price: None

--- LISTING 3 ---
Title: Used Cit

In [40]:
import requests
from bs4 import BeautifulSoup
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# -------------------------
# 1. SETUP: Input Data
# -------------------------
target_car_data = """
car type: Seat Toledo 1.9 TDI Signo
Build year: 2000
"""

# -------------------------
# 2. QUERY BUILDER (RANDOM SITES)
# -------------------------
def build_search_query(target_car_data):
    """
    Extracts car model, year, mileage & builds a search query 
    WITHOUT limiting to specific sites.
    """

    text = target_car_data

    # Extract model keywords
    model_line = re.search(r"car type:\s*(.+)", text, re.IGNORECASE)
    model_keywords = model_line.group(1) if model_line else ""

    # Extract year
    year = re.search(r"Build year:\s*(\d{4})", text)
    year_val = year.group(1) if year else ""

    # Extract mileage
    km_match = re.search(r"Odometer reading:\s*([\d, ]+)\s*km", text)
    if km_match:
        exact_km = int(km_match.group(1).replace(",", "").replace(" ", ""))
        mileage_range = f"{exact_km - 10000}-{exact_km + 10000} km"
    else:
        mileage_range = ""

    # 🔥 Build final search query (NO SITE FILTER)
    query = f'{model_keywords}" {year_val} "{mileage_range} price'

    return query.strip()

# -------------------------
# 3. SCRAPER (same)
# -------------------------
def extract_listing_data(url):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
            "(KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
        )
    }
    try:
        html = requests.get(url, headers=headers, timeout=30).text
        soup = BeautifulSoup(html, "html.parser")

        for tag in soup(["script", "style", "nav", "footer", "noscript"]):
            tag.extract()

        text = soup.get_text(" ", strip=True)
        text = re.sub(r"\s+", " ", text)
        return text[:7000]
    except Exception as e:
        return f"Failed to scrape {url}: {e}"

# -------------------------
# 4. MAIN: Search & Scrape
# -------------------------
TAVILY_KEY = "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"
TAVILY_URL = "https://api.tavily.com/search"

search_query = build_search_query(target_car_data)
print("🔍 Generated Search Query:")
print(search_query)

payload = {
    "api_key": TAVILY_KEY,
    "query": search_query,
    "search_depth": "basic",  # snippets only
    "max_results": 20,
    "include_answer": False
}

try:
    response = requests.post(TAVILY_URL, json=payload, timeout=30)
    response.raise_for_status()
    data = response.json()
except Exception as e:
    print("Search request failed:", e)
    data = {}

results = data.get("results", [])
print(f"\nFound {len(results)} results.\n")

market_context = ""

print("Starting deep scrapes...\n")

for i, r in enumerate(results[:40]):
    content = extract_listing_data(r["url"])
    market_context += f"\n--- LISTING {i+1} ---\n"
    market_context += f"Title: {r['title']}\n"
    market_context += f"URL: {r['url']}\n"
    market_context += f"CONTENT:\n{content[:5000]}...\n"

print(market_context)


🔍 Generated Search Query:
Seat Toledo 1.9 TDI Signo" 2000 " price

Found 14 results.

Starting deep scrapes...


--- LISTING 1 ---
Title: Car Seat Toledo Signo 1,8 20V Autom. , 949 EUR
URL: https://www.truck1-us.com/new-and-used/other/cars/seat-toledo-signo-1-8-20v-autom-a8558867.html
CONTENT:
Truck1 Loading......

--- LISTING 2 ---
Title: Find SEAT Toledo from 2000 for sale
URL: https://www.autoscout24.com/lst/seat/toledo/re_2000
CONTENT:
Find SEAT Toledo from 2000 for sale - AutoScout24 Skip to main content 10 Offers for SEAT Toledo produced in 2000 Sort: Best results Best results Price ascending Price descending Latest offers first Mileage ascending Mileage descending Power ascending Power descending First registration ascending First registration descending Filters 4 SEAT Toledo ⊕ Europe First registration from 2000 ⊕ First registration to 2000 ⊕ Remove all filters SEAT Toledo Toledo 2.3 V5 Add to list 7 € 1,750 01/2000 161,000 km Gasoline 110 kW (150 hp) Private seller DE-03185 Dr

In [19]:
search_query

'"Volvo V40 2.0 D3 Kinetic" 2015 "186801-188801 km" price'

In [11]:
import requests
from bs4 import BeautifulSoup
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# -------------------------
# 1. SETUP: Input Data
# -------------------------
target_car_data = """
Volvo V40 2.0 D3 Kinetic, 2015
Odometer reading:	187,801 km
"""

# -------------------------
# 2. QUERY BUILDER (FIXED)
# -------------------------
def build_search_query(target_car_data):
    """
    Extracts car model, year, mileage & builds a tight search query 
    for autoscout24.com or mobile.de.
    """

    text = target_car_data

    # 1. Extract model keywords
    model_line = re.search(r"car type:\s*(.+)", text, re.IGNORECASE)
    if model_line:
        model_keywords = model_line.group(1)
    else:
        model_keywords = ""

    # 2. Extract year
    year = re.search(r"Build year:\s*(\d{4})", text)
    year_val = year.group(1) if year else ""

    # 3. Extract mileage
    km_match = re.search(r"Odometer reading:\s*([\d, ]+)\s*km", text)
    if km_match:
        exact_km = int(km_match.group(1).replace(",", "").replace(" ", ""))
        start_km = exact_km - 1000
        end_km = exact_km + 1000
        mileage_range = f"{start_km}-{end_km} km"
    else:
        mileage_range = ""

    # 🔥 Build final search query
    query = (
        f'"{model_keywords}" {year_val} "{mileage_range}" '
        f"site:autoscout24.com OR site:mobile.de"
    )

    return query.strip()

# -------------------------
# 3. SCRAPER (same)
# -------------------------
def extract_listing_data(url):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    try:
        html = requests.get(url, headers=headers, timeout=30).text
        soup = BeautifulSoup(html, "html.parser")

        for tag in soup(["script", "style", "nav", "footer", "noscript"]):
            tag.extract()

        text = soup.get_text(" ", strip=True)
        text = re.sub(r"\s+", " ", text)
        return text[:7000]
    except Exception as e:
        return f"Failed to scrape {url}: {e}"

# -------------------------
# 4. MAIN: Search & Scrape
# -------------------------
TAVILY_KEY = "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"
TAVILY_URL = "https://api.tavily.com/search"

# Build final search query
search_query = build_search_query(target_car_data)
print("🔍 Generated Search Query:")
print(search_query)

payload = {
    "api_key": TAVILY_KEY,
    "query": search_query,
    "search_depth": "basic",
    "max_results": 20,
    "include_answer": False
}

try:
    response = requests.post(TAVILY_URL, json=payload, timeout=30)
    response.raise_for_status()
    data = response.json()
except Exception as e:
    print("Search request failed:", e)
    data = {}

results = data.get("results", [])
print(f"\nFound {len(results)} results.\n")

market_context = ""

print("Starting deep scrapes...\n")

for i, r in enumerate(results[:20]):
    content = extract_listing_data(r["url"])
    market_context += f"\n--- LISTING {i+1} ---\n"
    market_context += f"Title: {r['title']}\n"
    market_context += f"URL: {r['url']}\n"
    market_context += f"CONTENT:\n{content[:5000]}...\n"

print(market_context)


🔍 Generated Search Query:
""  "186801-188801 km" site:autoscout24.com OR site:mobile.de

Found 7 results.

Starting deep scrapes...


--- LISTING 1 ---
Title: Auto gebraucht kaufen bei mobile.de
URL: https://suchen.mobile.de/auto/auto.html
CONTENT:
Zugriff verweigert / Access denied Zugriff verweigert Access denied Reference Error: 0.7ee1502.1765300004.bc0d2259 Leider kÃ¶nnen wir Ihnen keinen Zugriff auf unsere Daten gewÃ¤hren. Wenn Sie Interesse an unseren Daten haben, kontaktieren Sie uns bitte: Unfortunately, automated access to this page was denied. If you are interested in accessing our data, please contact us: Telefon Montag bis Freitag von 8:00 - 18:00 Private Nutzer und gewerbliche Anbieter: +49 (0) 30 81097-601 HÃ¤ndler: +49 (0) 30 81097-500 Phone Mo. - Fr. 8:00 - 18:00 Private users and commercial sellers: +49 (0) 30 81097-601 Dealers: +49 (0) 30 81097-500 E-Mail service@team.mobile.de...

--- LISTING 2 ---
Title: Gebrauchtwagen von Privatanbietern kaufen bei mobile
URL: http

# Running the model

In [49]:
system_prompt = (f'Your role is to extract the necessary information from this context {market_context}'
                'only used realistic prices')

user_content = ('Get me the range of prices'
               )
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_content}
]

# -------------------------
# Apply chat template
# -------------------------
chat_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(chat_prompt, return_tensors="pt").to(model.device)

# -------------------------
# Generate Price
# -------------------------
generated = model.generate(
    **inputs,
    max_new_tokens=300,
    temperature=0.3
)

# Decode and print the output
full_ids = generated[0]

# Length of the input prompt
prompt_len = inputs["input_ids"].shape[1]

# Slice only the generated continuation
generated_only_ids = full_ids[prompt_len:]

# Decode only the new tokens
output = tokenizer.decode(generated_only_ids, skip_special_tokens=True)

print(output)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Based on the provided listings, here are the realistic prices for used Citroën Berlingo vehicles:

### Diesel:
- **1.5 BlueHDi 130** (Diesel):
  - **€ 13,900** (02/2019, 130,884 km)
  - **€ 16,950** (06/2020, 103,004 km)
  - **€ 18,740** (02/2019, 80,000 km)
  - **€ 17,440** (03/2020, 85,759 km)
  - **€ 16,912** (10/2021, 99,945 km)

### Gasoline:
- **1.2 PureTech 110** (Gasoline):
  - **€ 11,450** (06/2019, 99,986 km)
  - **€ 10,970** (06/2016, 84,687 km)
  - **€ 10,499** (01/2018, 143,661


In [50]:

system_prompt = (
    "We are in December 2025. "
    "You are an expert European vehicle appraiser. "
    "Your only task is to output a final estimated price range in EUR (low-high). "
    "IMPORTANT RULES:\n"
    f"1. You must use the prices inside the {output} and cite the average price"
    "2. Your explanation MUST reference at least two values from market_context. "
    "3. Apply penalties strictly:\n"
    "   - Age : -30% to -40%\n"
    "   - High mileage : -15% to -25%\n"
    "   - EURO 4 diesel: -10% to -20%\n"
    "   - Accident history: -500 to -1000 EUR\n"
    "   - Multiple repaints: -300 to -600 EUR\n"
    "   - Interior wear: -200 EUR\n"
    "   - Only 1 key: -150 EUR\n"
    "   - Missing COC: -200 EUR\n"
    "4. The final price range must be realistic, tight, and cannot exceed -20% difference.\n"
    "5. Do NOT use general car knowledge. ONLY use the provided market_context.\n"
    "6. Old diesel sedans with Euro 4, high mileage, damages, and 5 owners must always be valued in the low segment.\n"
    "Explain in exactly three sentences why you chose this price range."
)




user_content = f"""
I want a range of price for this car from low-end to high-end:
car type: Citroen Berlingo 1.5 Blue-HDi Club M 1000
Build year:	2019
First registration:	08/2019
Odometer reading:	115,077 km
Fuel type:	Diesel
Horsepower:	75 kW / 102 HP
Cylinder capacity:	1,499 ccm
Gear box:	Manual
Inspection expires:	05/2026
Body type:	Chassis
Total number of owners:	3  
Keys:	1
Prior damage / Accident according to previous owner	Yes
Country of origin:	FR
Country of last registration:	FR
Environmental class:	EURO 6
COC papers:	No
Seats:	2
Color:	White (Original)
Upholstery:	Cloth (Original)
Door count:	4
CO2 Emissions:	109g/km

Damage summary:
Area	Damage	Severity	Description	Quantity	Parts affected
Exterior	Crack	-	-	1	Rear left light (1)
Dent	Width: ≥50mm	Paint damage	4	Rear right door (2), Right side skirts (2)
Width: ≥5mm & <20mm	No paint damage	2	Front left door (1), Rear right fender (1)
Scratch	Length: ≥100mm	Deep scratch - down to primer	2	Tailgate (2)
Length: ≥20mm & <50mm	Deep scratch - down to primer	6	Front bumper (3), Right mirror housing (3)
Stone chip	Width: ≤3mm	Paint damage	4	Hood (4)
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_content}
]

# -------------------------
# Apply chat template
# -------------------------
chat_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(chat_prompt, return_tensors="pt").to(model.device)

# -------------------------
# Generate Price
# -------------------------
generated = model.generate(
    **inputs,
    max_new_tokens=300,
    temperature=0.3
)

# Decode and print the output
full_ids = generated[0]

# Length of the input prompt
prompt_len = inputs["input_ids"].shape[1]

# Slice only the generated continuation
generated_only_ids = full_ids[prompt_len:]

# Decode only the new tokens
output = tokenizer.decode(generated_only_ids, skip_special_tokens=True)

print(output)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Based on the provided listings and the car's condition, the estimated price range for the Citroën Berlingo 1.5 BlueHDi Club M 1000 (2019) with 115,077 km is:

**Low-end: €12,000**
**High-end: €14,000**

**Explanation:**
- The car is a 2019 model with 115,077 km, which is relatively high mileage, warranting a discount.
- The car has some exterior damage, including cracks, dents, scratches, and stone chips, which will also reduce its value.
- The car has only one key and no COC papers, which are additional factors that will lower the price.
- The car is a diesel vehicle with Euro 6 emissions, which is favorable, but the high mileage and damage will offset this advantage.


In [43]:
system_prompt = (f'Your role is to extract the necessary information from this context {market_context}'
                'only used realistic prices')

user_content = ('Get me the range of prices'
               )
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_content}
]

# -------------------------
# Apply chat template
# -------------------------
chat_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(chat_prompt, return_tensors="pt").to(model.device)

# -------------------------
# Generate Price
# -------------------------
generated = model.generate(
    **inputs,
    max_new_tokens=300,
    temperature=0.3
)

# Decode and print the output
full_ids = generated[0]

# Length of the input prompt
prompt_len = inputs["input_ids"].shape[1]

# Slice only the generated continuation
generated_only_ids = full_ids[prompt_len:]

# Decode only the new tokens
output = tokenizer.decode(generated_only_ids, skip_special_tokens=True)

print(output)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Based on the provided listings, here are the ranges of prices for the Volvo C30 models:

### Volvo C30 d5:
- **Price Range:** €2,900 - €10,999

### Volvo C30 t5:
- **Price Range:** €5,500 - €18,499

### Summary:
- **Volvo C30 d5:** €2,900 - €10,999
- **Volvo C30 t5:** €5,500 - €18,499


In [44]:

system_prompt = (
    "We are in December 2025. "
    "You are an expert European vehicle appraiser. "
    "Your only task is to output a final estimated price range in EUR (low-high). "
    "IMPORTANT RULES:\n"
    f"1. You must use the prices inside the {output} and cite the average price"
    "2. Your explanation MUST reference at least two values from market_context. "
    "3. Apply penalties strictly:\n"
    "   - Age : -30% to -40%\n"
    "   - High mileage : -15% to -25%\n"
    "   - EURO 4 diesel: -10% to -20%\n"
    "   - Accident history: -500 to -1000 EUR\n"
    "   - Multiple repaints: -300 to -600 EUR\n"
    "   - Interior wear: -200 EUR\n"
    "   - Only 1 key: -150 EUR\n"
    "   - Missing COC: -200 EUR\n"
    "4. The final price range must be realistic, tight, and cannot exceed -20% difference.\n"
    "5. Do NOT use general car knowledge. ONLY use the provided market_context.\n"
    "6. Old diesel sedans with Euro 4, high mileage, damages, and 5 owners must always be valued in the low segment.\n"
    "Explain in exactly three sentences why you chose this price range."
)




user_content = f"""
I want a range of price for this car from low-end to high-end:
car type: Volvo C30 2.4 D5 Summum
Build year:	2006
First registration:	01/2007
Odometer reading:	191,419 km
Fuel type:	Diesel
Horsepower:	132 kW / 180 HP
Cylinder capacity:	2,400 ccm
Gear box:	Automatic
Inspection expires:	10/2026
Body type:	Sedan
Total number of owners:	5   (Including car dealers)
Keys:	1
Prior damage / Accident according to previous owner	Yes
Country of origin:	ES
Country of last registration:	ES
Environmental class:	EURO 4
COC papers:	No
Seats:	4
Color:	Gray (Original)
Upholstery:	Leather (Original)
Door count:	3

Damage summary:
Area	Damage	Severity	Description	Quantity	Parts affected
Exterior	Dent	Width: ≥5mm & <20mm	No paint damage	1	Rear right fender (1)
Repaint	Thickness: ≥250µm & <500µm	Professional repaint	3	Front left fender (1), Rear left fender (1), Front right fender (1)
Thickness: ≥125µm & <250µm	Professional repaint	3	Hood (1), Front left door (1), Front right door (1)
Scratch	Length: ≥100mm	Deep scratch - down to primer	1	Rear bumper (1)
Length: ≥50mm & <100mm	Deep scratch - down to primer	1	Rear right fender (1)
Length: ≥20mm & <50mm	Surface scratch	3	Front left door (1), Front right door (2)
Length: ≥20mm & <50mm	Deep scratch - down to primer	1	Front bumper (1)
Length: <20mm	Deep scratch - down to primer	6	Front left door (2), Front right door (2), Front right door handle (2)
Stone chip	Width: ≤3mm	Paint damage	2	Hood (2)
Interior	Worn out	-	-	6	Front door interior left (1), Driver seat (1), Steering Wheel (1), Center console (1), Dashboard (1), Front door interior right (1)
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_content}
]

# -------------------------
# Apply chat template
# -------------------------
chat_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(chat_prompt, return_tensors="pt").to(model.device)

# -------------------------
# Generate Price
# -------------------------
generated = model.generate(
    **inputs,
    max_new_tokens=300,
    temperature=0.3
)

# Decode and print the output
full_ids = generated[0]

# Length of the input prompt
prompt_len = inputs["input_ids"].shape[1]

# Slice only the generated continuation
generated_only_ids = full_ids[prompt_len:]

# Decode only the new tokens
output = tokenizer.decode(generated_only_ids, skip_special_tokens=True)

print(output)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Based on the provided information, the Volvo C30 2.4 D5 Summum from 2006 with 191,419 km, Euro 4 diesel, and multiple damages, the estimated price range is €1,500 - €3,500.

### Explanation:
1. **Age and Mileage Penalties:** The car is 19 years old and has 191,419 km. This results in a significant age penalty of -30% to -40% and a high mileage penalty of -15% to -25%.
2. **Euro 4 Diesel Penalty:** The car is a Euro 4 diesel, which incurs a penalty of -10% to -20%.
3. **Damage and Repaint Penalties:** The car has multiple damages, including dents, scratches, and repaints, which add up to a penalty of -1,000 EUR to -2,000 EUR. Additionally, the missing COC papers and interior wear contribute to a further reduction of -200 EUR and -200 EUR, respectively.

Given these factors, the final estimated price range is €1,500 - €3,500.


In [52]:
system_prompt = (f'Your role is to extract the necessary information from this context {market_context}'
                'only used realistic prices')

user_content = ('Get me the range of prices'
               )
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_content}
]

# -------------------------
# Apply chat template
# -------------------------
chat_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(chat_prompt, return_tensors="pt").to(model.device)

# -------------------------
# Generate Price
# -------------------------
generated = model.generate(
    **inputs,
    max_new_tokens=300,
    temperature=0.3
)

# Decode and print the output
full_ids = generated[0]

# Length of the input prompt
prompt_len = inputs["input_ids"].shape[1]

# Slice only the generated continuation
generated_only_ids = full_ids[prompt_len:]

# Decode only the new tokens
output = tokenizer.decode(generated_only_ids, skip_special_tokens=True)

print(output)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Based on the provided listings, here are the realistic prices for used Volkswagen Tiguan Allspace vehicles:

1. **Volkswagen Tiguan Allspace 2.0 TDI SCR DSG 4MOTION Advanced BMT** - €23,000
2. **Volkswagen Tiguan Allspace 2.0 TDI SCR 4Motion DSG Highline*AHK*Stan** - €32,480
3. **Volkswagen Tiguan Allspace 2.0 tdi scr Life 150cv 7p.ti dsg** - €33,900
4. **Volkswagen Tiguan ALLSPACE 2.0 TDi SCR 7plazas DSG** - €24,890
5. **Volkswagen Tiguan Allspace 2.0 TDI SCR 4Motion DSG** - €36,480
6. **Volkswagen Tiguan Allspace 2.0 TDI SCR DSG** - €26,980
7. **Volkswagen Tiguan Allspace 2.0 TDI SCR 4Motion DSG Highline*AHK*Pano** - €22,480
8. **Volkswagen Tiguan Tiguan Allspace 2.0 TDI 150 CV SCR D


In [53]:

system_prompt = (
    "We are in December 2025. "
    "You are an expert European vehicle appraiser. "
    "Your only task is to output a final estimated price range in EUR (low-high). "
    "IMPORTANT RULES:\n"
    f"1. You must use the prices inside the {output} and cite the average price"
    "2. Your explanation MUST reference at least two values from market_context. "
    "3. Apply penalties strictly:\n"
    "   - Age : -30% to -40%\n"
    "   - High mileage : -15% to -25%\n"
    "   - EURO 4 diesel: -10% to -20%\n"
    "   - Accident history: -500 to -1000 EUR\n"
    "   - Multiple repaints: -300 to -600 EUR\n"
    "   - Interior wear: -200 EUR\n"
    "   - Only 1 key: -150 EUR\n"
    "   - Missing COC: -200 EUR\n"
    "4. The final price range must be realistic, tight, and cannot exceed -20% difference.\n"
    "5. Do NOT use general car knowledge. ONLY use the provided market_context.\n"
    "6. Old diesel sedans with Euro 4, high mileage, damages, and 5 owners must always be valued in the low segment.\n"
    "Explain in exactly three sentences why you chose this price range."
)




user_content = f"""
I want a range of price for this car from low-end to high-end:
car type: Volkswagen Tiguan Allspace 2.0 TDI Carat
Build year:	2018
First registration:	11/2018
Odometer reading:	123,239 km
Fuel type:	Diesel
Horsepower:	110 kW / 150 HP
Cylinder capacity:	1,968 ccm
Gear box:	Duplex
Inspection expires:	11/2026
Body type:	SUV
Total number of owners:	1  
Keys:	2
Prior damage / Accident according to previous owner	No
Country of origin:	BE
Country of last registration:	BE
Environmental class:	EURO 6d-TEMP
COC papers:	Yes
Seats:	5
Color:	Gray (Original)
Upholstery:	Leather (Original)
Door count:	5
CO2 Emissions:	168g/km

Damage summary:
Area	Damage	Severity	Description	Quantity	Parts affected
Exterior	Dent	Width: <5mm	Paint damage	2	Rear left door (1), Front right door (1)
Width: <5mm	No paint damage	1	Front left door (1)
Scratch	Length: ≥100mm	Deep scratch - down to primer	1	Front right door (1)
Length: ≥20mm & <50mm	Surface scratch	1	Front bumper (1)
Length: <20mm	Surface scratch	1	Rear right fender (1)
Length: <20mm	Deep scratch - down to primer	2	Rear bumper (2)
Stone chip	Width: ≤3mm	In driver's eyeline	3	Windscreen (3)
Width: ≤3mm	Paint damage	5	Hood (2), Rear left fender (2), Rear bumper (1)
Interior	Reconditioning necessary	-	-	3	Door interior rear right (1), Driver seat (1), Steering Wheel (1)
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_content}
]

# -------------------------
# Apply chat template
# -------------------------
chat_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(chat_prompt, return_tensors="pt").to(model.device)

# -------------------------
# Generate Price
# -------------------------
generated = model.generate(
    **inputs,
    max_new_tokens=300,
    temperature=0.3
)

# Decode and print the output
full_ids = generated[0]

# Length of the input prompt
prompt_len = inputs["input_ids"].shape[1]

# Slice only the generated continuation
generated_only_ids = full_ids[prompt_len:]

# Decode only the new tokens
output = tokenizer.decode(generated_only_ids, skip_special_tokens=True)

print(output)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Based on the provided market context and the specific details of the Volkswagen Tiguan Allspace 2.0 TDI Carat, the estimated price range for this vehicle in December 2025 would be:

**Low-end: €20,000 - High-end: €25,000**

**Explanation:**
1. The vehicle is a 2018 model with 123,239 km on the odometer, which is relatively high mileage, warranting a discount.
2. The vehicle has some exterior damage, including dents, scratches, and stone chips, which will reduce its value.
3. The interior requires reconditioning, which will also impact the price.


In [43]:
system_prompt = (f'Your role is to extract the necessary information from this context {market_context}'
                )

user_content = ('Get me the range of prices lower end-higher-end and give me the average.'
               )
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_content}
]

# -------------------------
# Apply chat template
# -------------------------
chat_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(chat_prompt, return_tensors="pt").to(model.device)

# -------------------------
# Generate Price
# -------------------------
generated = model.generate(
    **inputs,
    max_new_tokens=300,
    temperature=0.3
)

# Decode and print the output
full_ids = generated[0]

# Length of the input prompt
prompt_len = inputs["input_ids"].shape[1]

# Slice only the generated continuation
generated_only_ids = full_ids[prompt_len:]

# Decode only the new tokens
prices = tokenizer.decode(generated_only_ids, skip_special_tokens=True)

print(prices)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Based on the provided listings, here are the ranges and average prices for the SEAT Toledo models:

### SEAT Toledo 1.8 20V Automatik
- **Lower End:** €499
- **Higher End:** €4,250
- **Average:** (€499 + €4,250) / 2 = €2,374.50

### SEAT Toledo 1.9 TDi 90 Stella
- **Lower End:** €950
- **Higher End:** €2,450
- **Average:** (€950 + €2,450) / 2 = €1,700

### SEAT Toledo 1.9 TDi 110 CH
- **Lower End:** €1,750
- **Higher End:** €2,450
- **Average:** (€1,750 + €2,450) / 2 = €2,100

### SEAT Toledo 1.6i Comfort
- **Lower End:** €1,450
- **Higher End:** €1,750
- **Average:** (€1,450 + €1,750) / 2 = €1,600

### SEAT Toledo 1.


In [44]:

system_prompt = (
    "We are in December 2025. "
    "You are an expert European vehicle appraiser. "
    "Your only task is to output a final estimated price range in EUR (low-high). "
    "IMPORTANT RULES:\n"
    f"1. You must use the prices inside the {prices} and use the average to define the lower higher end. "
    "2. Your explanation MUST reference at least two values from market_context. "
    "3. Apply penalties strictly:\n"
    "   - Age : -30% to -40%\n"
    "   - High mileage : -15% to -25%\n"
    "   - EURO 4 diesel: -10% to -20%\n"
    "   - Accident history: -500 to -1000 EUR\n"
    "   - Multiple repaints: -300 to -600 EUR\n"
    "   - Interior wear: -200 EUR\n"
    "   - Only 1 key: -150 EUR\n"
    "   - Missing COC: -200 EUR\n"
    "4. The final price range must be realistic, tight, and cannot exceed +10% difference. - High quality maker add value depending on the maker can exceed +20%\n"
    "5. Do NOT use general car knowledge. ONLY use the provided market_context.\n"
    "6. Old diesel sedans with Euro 4, high mileage, damages, and 5 owners must always be valued in the low segment.\n"
    "Explain in exactly three sentences why you chose this price range."
    "7. Finally highlight the final price range in a totally seperate new paragraph and type: 'the final price range is: {price range}. "
)




user_content = f"""
I want a range of price for this car from low-end to high-end:
car type: BMW X1 sDrive 18d Sport Line
Build year:	2021
First registration:	04/2021
Odometer reading:	72,699 km
Fuel type:	Diesel
Horsepower:	110 kW / 150 HP
Cylinder capacity:	1,995 ccm
Gear box:	Manual
Inspection expires:	03/2027
Body type:	SUV
Total number of owners:	1   (Including car dealers)
Keys:	2
Prior damage / Accident according to previous owner	No
Country of origin:	IT
Country of last registration:	IT
Environmental class:	EURO 6d
COC papers:	No
Seats:	5
Color:	White (Original)
Upholstery:	Cloth (Original)
Door count:	5
CO2 Emissions:	132g/km

Area	Damage	Severity	Description	Quantity	Parts affected
Exterior	Scratch	Length: ≥20mm & <50mm	Deep scratch - down to primer	6	Front bumper (2), Front left door (1), Rear bumper (2), Front right door (1)
Length: <25mm	Deep scratch	2	Rear left rim (1), Front right rim (1)
Length: <20mm	Deep scratch - down to primer	2	Rear right fender (2)
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_content}
]

# -------------------------
# Apply chat template
# -------------------------
chat_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(chat_prompt, return_tensors="pt").to(model.device)

# -------------------------
# Generate Price
# -------------------------
generated = model.generate(
    **inputs,
    max_new_tokens=300,
    temperature=0.3
)

# Decode and print the output
full_ids = generated[0]

# Length of the input prompt
prompt_len = inputs["input_ids"].shape[1]

# Slice only the generated continuation
generated_only_ids = full_ids[prompt_len:]

# Decode only the new tokens
output = tokenizer.decode(generated_only_ids, skip_special_tokens=True)

print(output)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Based on the provided market context and the specific details of the BMW X1 sDrive 18d Sport Line, the car's age, mileage, and condition suggest a significant discount. The car is a 2021 model with 72,699 km, which is relatively high for a 2021 vehicle. The absence of COC papers and the presence of multiple scratches on the exterior, including deep scratches down to primer, indicate a need for a substantial discount. Additionally, the car's environmental class is EURO 6d, which is more recent and thus less penalized compared to EURO 4 diesel cars. Given these factors, the estimated price range is:

**Final Price Range:** €10,000 - €12,000

The final price range is: €10,000 - €12,000


In [45]:
system_prompt = (f'Your role is to extract the necessary information from this context {market_context}'
                )

user_content = ('Get me the range of prices lower end-higher-end and give me the average all in euro. '
               )
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_content}
]

# -------------------------
# Apply chat template
# -------------------------
chat_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(chat_prompt, return_tensors="pt").to(model.device)

# -------------------------
# Generate Price
# -------------------------
generated = model.generate(
    **inputs,
    max_new_tokens=300,
    temperature=0.3
)

# Decode and print the output
full_ids = generated[0]

# Length of the input prompt
prompt_len = inputs["input_ids"].shape[1]

# Slice only the generated continuation
generated_only_ids = full_ids[prompt_len:]

# Decode only the new tokens
prices = tokenizer.decode(generated_only_ids, skip_special_tokens=True)

print(prices)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Based on the provided listings, here are the ranges of prices for the SEAT Toledo models:

### Lower End:
- **SEAT Toledo 1.6i Comfort** - €1,450
- **SEAT Toledo 1.6i+Comfort** - €1,450
- **SEAT Toledo 1.6-16V Stella** - €2,450
- **SEAT Toledo 1.6-16V Sport** - €2,450
- **SEAT Toledo 1.6-16V Signo** - €2,450
- **SEAT Toledo 1.6-16V Signo** - €2,450
- **SEAT Toledo 1.6-16V Signo** - €2,450
- **SEAT Toledo 1.6-16V Signo** - €2,450
- **SEAT Toledo 1.6-16V Signo** - €2,450
- **SEAT Toledo 1.6-16V Signo** - €2,450
- **SEAT Toledo 1.6-16V Signo** - €2,450
- **SEAT Toledo 1.6-16V Signo** - €2,


In [49]:

system_prompt = (
    "We are in December 2025. "
    "You are an expert European vehicle appraiser. "
    "Your only task is to output a final estimated price range in EUR (low-high). "
    "IMPORTANT RULES:\n"
    f"1. You must use the prices inside the {prices} and use the average to define the lower higher end. "
    "2. Your explanation MUST reference at least two values from market_context. "
    "3. Apply penalties strictly:\n"
    "   - Age : -30% to -40%\n"
    "   - High mileage : -20% to 25%\n"
    "   - EURO 4 diesel: -10% to -20%\n"
    "   - Accident history: -500 to -1000 EUR\n"
    "   - Multiple repaints: -300 to -600 EUR\n"
    "   - Interior wear: -200 EUR\n"
    "   - Only 1 key: -150 EUR\n"
    "   - Missing COC: -200 EUR\n"
    "4. The final price range must be realistic, tight, and cannot exceed +10% difference. - High quality maker add value depending on the maker can exceed +20%\n"
    "5. Do NOT use general car knowledge. ONLY use the provided market_context.\n"
    "6. Old diesel sedans with Euro 4, high mileage, damages, and 5 owners must always be valued in the low segment.\n"
    "Explain in exactly three sentences why you chose this price range."
    "7. Finally highlight the final price range in a totally seperate new paragraph and type: 'the final price range is: {price range}. "
    "8. the difference between the lower and higher end shouldn't exceed 3000"
)




user_content = f"""
What's the price of this car? I want a range of price for this car from low-end to high-end:
car type: Seat Toledo 1.9 TDI Signo
Build year:	2000
First registration:	06/2000
Odometer reading:	206,478 km
Fuel type:	Diesel
Horsepower:	81 kW / 110 HP
Cylinder capacity:	1,896 ccm
Gear box:	Manual
Inspection expires:	07/2026
Body type:	Sedan
Total number of owners:	2   (Including car dealers)
Keys:	2
Prior damage / Accident according to previous owner	No
Country of origin:	ES
Country of last registration:	ES
Environmental class:	N/A
COC papers:	No
Seats:	5
Color:	Blue (Original)
Upholstery:	Cloth (Original)
Door count:	4


Area	Damage	Severity	Description	Quantity	Parts affected
Exterior	Dent	Width: ≥20mm & <50mm	No paint damage	2	Front bumper (1), Rear left door (1)
Width: ≥5mm & <20mm	Paint damage	3	Front bumper (20), Rear right door (12)
Width: <5mm	Paint damage	1	Front right door (7)
Fracture	Length: ≥20mm & <50mm, Width: <5mm	-	1	Left mirror housing (1)
Scratch	Length: ≥100mm	Deep scratch - down to primer	1	Front bumper (4)
Length: ≥20mm & <50mm	Deep scratch - down to primer	5	Front left fender (2), Rear bumper (3)
Length: ≥20mm & <50mm	Surface scratch	9	Hood (12), Front left door (14), Rear left door (8), Rear right fender (1), Rear right door (4)
Length: <20mm	Surface scratch	4	Front right door (6), Roof (9)
Length: <20mm	Deep scratch - down to primer	6	Rear left door (6), Rear left fender (11), Rear right fender (5), Front right door handle (1), Front right fender (1)
Stone chip	Width: ≤3mm	In driver's eyeline	3	Windscreen (3)
Width: ≤3mm	Paint damage	7	Hood (21), Front left fender (5), Rear right door (18), Front right fender (2)
Interior	Dirty	-	-	1	Seat back left (1)
Reconditioning necessary	-	-	8	Front door interior right (4), Passenger seat (1), Dashboard (15), Seat back right (1), Front door interior left (1), Driver seat (1), Safety belt driver (1), Hat rack (1)
Scratch	Length: <50mm	-	1	Door interior rear right (10)
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_content}
]

# -------------------------
# Apply chat template
# -------------------------
chat_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(chat_prompt, return_tensors="pt").to(model.device)

# -------------------------
# Generate Price
# -------------------------
generated = model.generate(
    **inputs,
    max_new_tokens=300,
    temperature=0.3
)

# Decode and print the output
full_ids = generated[0]

# Length of the input prompt
prompt_len = inputs["input_ids"].shape[1]

# Slice only the generated continuation
generated_only_ids = full_ids[prompt_len:]

# Decode only the new tokens
output = tokenizer.decode(generated_only_ids, skip_special_tokens=True)

print(output)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Based on the provided market context and the condition of the car, the SEAT Toledo 1.9 TDI Signo from 2000 with 206,478 km on the odometer, and considering the damages and the number of owners, the car falls into the lower segment. The car has significant exterior damage, including dents, scratches, and paint damage, which will significantly reduce its value. Additionally, the absence of COC papers and the high mileage will further depreciate the car's value. Given these factors, the estimated price range for this car would be between €1,000 and €1,500.

The final price range is: €1,000 - €1,500.
